In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from typing import List
from tqdm import tqdm
import random
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

c:\Users\kanze\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_users = pd.read_csv('data_original/data_original/users.csv')
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_items = pd.read_csv('data_original/data_original/items.csv')

In [3]:
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')

In [4]:
df_interactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5476251 entries, 0 to 5476250
Data columns (total 5 columns):
 #   Column         Dtype         
---  ------         -----         
 0   user_id        int64         
 1   item_id        int64         
 2   last_watch_dt  datetime64[ns]
 3   total_dur      int64         
 4   watched_pct    float64       
dtypes: datetime64[ns](1), float64(1), int64(3)
memory usage: 208.9 MB


In [5]:
df_interactions['completed'] = (df_interactions['watched_pct'] >= 60).astype(int)

In [6]:
last_interactions = df_interactions.tail(10)

In [7]:
last_interactions

,user_id,item_id,last_watch_dt,total_dur,watched_pct,completed
5476241,1073802,9927,2021-08-07,6425,97.0,1
5476242,268216,3071,2021-04-21,5752,98.0,1
5476243,497899,9629,2021-05-29,45,1.0,0
5476244,438585,7829,2021-08-02,6804,100.0,1
5476245,786732,4880,2021-05-12,753,0.0,0
5476246,648596,12225,2021-08-13,76,0.0,0
5476247,546862,9673,2021-04-13,2308,49.0,0
5476248,697262,15297,2021-08-20,18307,63.0,1
5476249,384202,16197,2021-04-19,6203,100.0,1
5476250,319709,4436,2021-08-15,3921,45.0,0


In [8]:
user_ids = last_interactions['user_id']
item_ids = last_interactions['item_id']
complete = last_interactions['completed']

In [9]:
matrix = csr_matrix((complete, (user_ids, item_ids)))
print(matrix)

<Compressed Sparse Row sparse matrix of dtype 'int32'
	with 10 stored elements and shape (1073803, 16198)>
  Coords	Values
  (268216, 3071)	1
  (319709, 4436)	0
  (384202, 16197)	1
  (438585, 7829)	1
  (497899, 9629)	0
  (546862, 9673)	0
  (648596, 12225)	0
  (697262, 15297)	1
  (786732, 4880)	0
  (1073802, 9927)	1


In [10]:
class ALS:
    def __init__(self, n_factors=10, alpha=40, regularization=0.1, iterations=15):
        """
        Инициализация ALS модели
        
        Parameters:
        - n_factors: количество латентных факторов
        - alpha: параметр уверенности для неявной обратной связи
        - regularization: коэффициент регуляризации (lambda)
        - iterations: количество итераций обучения
        """
        self.n_factors = n_factors
        self.alpha = alpha
        self.regularization = regularization
        self.iterations = iterations
        self.user_factors = None
        self.item_factors = None
        self.loss_history = []
    
    def fit(self, ratings):
        """
        Обучение ALS модели
        
        Parameters:
        - ratings: scipy sparse matrix в формате (users × items)
        """
        n_users, n_items = ratings.shape
        
        # Инициализация матриц факторов
        self.user_factors = np.random.normal(0, 0.1, (n_users, self.n_factors))
        self.item_factors = np.random.normal(0, 0.1, (n_items, self.n_factors))
        
        print("Начинаем обучение ALS...")
        
        for iteration in range(self.iterations):
            # Шаг 1: Фиксируем item factors, обновляем user factors
            for u in range(n_users):
                # Получаем индексы и значения оценок пользователя
                user_ratings = ratings[u].toarray().flatten()
                rated_items = np.where(user_ratings > 0)[0]
                
                if len(rated_items) > 0:
                    # Вычисляем уверенности для неявной обратной связи
                    confidence = 1 + self.alpha * user_ratings[rated_items]
                    
                    # Матрица item factors для оцененных items
                    Y = self.item_factors[rated_items]
                    
                    # Вычисляем матрицу A и вектор b
                    C = np.diag(confidence)
                    A = Y.T @ C @ Y + self.regularization * np.eye(self.n_factors)
                    b = Y.T @ confidence
                    
                    # Решаем линейную систему
                    self.user_factors[u] = np.linalg.solve(A, b)
            
            # Шаг 2: Фиксируем user factors, обновляем item factors
            for i in range(n_items):
                # Получаем индексы и значения оценок для item
                item_ratings = ratings[:, i].toarray().flatten()
                rating_users = np.where(item_ratings > 0)[0]
                
                if len(rating_users) > 0:
                    # Вычисляем уверенности
                    confidence = 1 + self.alpha * item_ratings[rating_users]
                    
                    # Матрица user factors для оценивших пользователей
                    X = self.user_factors[rating_users]
                    
                    # Вычисляем матрицу A и вектор b
                    C = np.diag(confidence)
                    A = X.T @ C @ X + self.regularization * np.eye(self.n_factors)
                    b = X.T @ confidence
                    
                    # Решаем линейную систему
                    self.item_factors[i] = np.linalg.solve(A, b)
            
            # Вычисляем loss
            loss = self._compute_loss(ratings)
            self.loss_history.append(loss)
            print(f"Итерация {iteration + 1}/{self.iterations}, Loss: {loss:.4f}")
    
    def _compute_loss(self, ratings):
        """Вычисление функции потерь"""
        loss = 0
        n_users, n_items = ratings.shape
        
        for u in range(n_users):
            for i in range(n_items):
                if ratings[u, i] > 0:
                    prediction = self.user_factors[u] @ self.item_factors[i]
                    confidence = 1 + self.alpha * ratings[u, i]
                    error = confidence - prediction
                    loss += error ** 2
        
        # Добавляем регуляризацию
        reg_loss = self.regularization * (
            np.sum(self.user_factors ** 2) + np.sum(self.item_factors ** 2)
        )
        
        return loss + reg_loss
    
    def predict(self, user_id, item_id):
        """Предсказание оценки"""
        return self.user_factors[user_id] @ self.item_factors[item_id]
    
    def recommend(self, user_id, n_recommendations=10):
        """Рекомендации для пользователя"""
        user_vector = self.user_factors[user_id]
        scores = user_vector @ self.item_factors.T
        top_items = np.argsort(scores)[::-1][:n_recommendations]
        return top_items, scores[top_items]
    
    def plot_loss(self):
        """Визуализация истории потерь"""
        plt.figure(figsize=(10, 6))
        plt.plot(self.loss_history, marker='o')
        plt.title('История обучения ALS')
        plt.xlabel('Итерация')
        plt.ylabel('Loss')
        plt.grid(True)
        plt.show()


In [18]:
# Надо написать самому
als = ALS()

In [19]:
print(als)

In [21]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# После обучения ALS создаём матрицы сходства
user_similarity = cosine_similarity(als.user_factors)
item_similarity = cosine_similarity(als.item_factors)

userKNN = pd.DataFrame(user_similarity)
itemKNN = pd.DataFrame(item_similarity)

print('Матрица сходства пользователей (userKNN):')
display(userKNN.head())
print('Матрица сходства предметов (itemKNN):')
display(itemKNN.head())


ValueError: Expected 2D array, got scalar array instead:
array=nan.
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [ ]:
import numpy as np

def apk(actual, predicted, k=10):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
    if not actual:
        return 0.0
    return score / min(len(actual), k)

def mapk(actual, predicted, k=10):
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

def get_top_k_recommendations(model, ratings, k=10):
    user_factors = model.user_factors
    item_factors = model.item_factors
    scores = user_factors @ item_factors.T
    top_k_items = np.argsort(-scores, axis=1)[:, :k]
    return top_k_items

top_k = get_top_k_recommendations(als, matrix, k=10)
actual = [matrix[u].nonzero()[1] for u in range(matrix.shape[0])]
predicted = [top_k[u] for u in range(matrix.shape[0])]

map10 = mapk(actual, predicted, k=10)
print(f'MAP@10: {map10:.4f}')
